<a href="https://colab.research.google.com/github/Ashprakash/groundfin/blob/main/benchmark/groundfin_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GROUNDFIN Colab Runner

This notebook is meant to stay stable. It pulls the latest code from GitHub, so most fixes should happen in `.py` files instead of requiring notebook replacement.

## 1. Pull Latest Project Code

Rerun this cell whenever the repo changes.

In [ ]:
%cd /content
!test -d groundfin/.git && git -C groundfin pull || git clone https://github.com/Ashprakash/groundfin.git
%cd /content/groundfin
!pip -q install -r requirements-colab.txt
!git log --oneline -1


## 2. Load FinanceBench

In [ ]:
import importlib
from benchmark import financebench_pilot as pilot

importlib.reload(pilot)
df = pilot.load_financebench()
pilot.summarize_dataset(df)
display(df.head(3))

## 3. Inspect Prompt

In [ ]:
print(pilot.make_prompt(df.iloc[0], with_evidence=True)[:2500])

## 4. Optional OpenAI Baseline

Set `OPENAI_API_KEY` in Colab secrets or uncomment the environment line below.

In [ ]:
import os

# os.environ['OPENAI_API_KEY'] = 'your_key_here'

if os.environ.get('OPENAI_API_KEY'):
    results, summary = pilot.run_openai_baseline(df, n_examples=20, model='gpt-4.1-mini')
    display(summary)
    display(results.head())
else:
    print('OPENAI_API_KEY is not set. Skipping API baseline.')

## 5. Open-Model Baseline

Use this when you do not have an API key. For best speed, set Colab to `Runtime -> Change runtime type -> T4 GPU`. The default model is small so we can validate the pipeline quickly.


In [ ]:
HF_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
N_EXAMPLES = 5

hf_results, hf_summary = pilot.run_hf_baseline(
    df,
    n_examples=N_EXAMPLES,
    model_id=HF_MODEL_ID,
    max_new_tokens=160,
    max_evidence_chars=6000,
)

hf_summary_reset = hf_summary.reset_index()
hf_preview = hf_results[['condition', 'gold_answer', 'parsed_answer', 'weak_match_answer', 'numeric_match_answer', 'refusal', 'prediction']].head(10)

print('=== HF SUMMARY ===')
print(hf_summary_reset.to_string(index=False))
print('\n=== HF PREVIEW ===')
print(hf_preview.to_string(index=False))

display(hf_summary)
display(hf_preview)

hf_summary_reset.to_csv('hf_summary.csv', index=False)
hf_results.to_csv('hf_results.csv', index=False)
print('\nWrote hf_summary.csv and hf_results.csv')


## 6. Grounding Probe

This directly tests the GROUNDFIN idea with four conditions: gold evidence, missing evidence, direct grounded evidence, and direct counterfactual evidence. Start tiny; this is a probe, not paper-grade evaluation yet.


In [ ]:
PROBE_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
N_PROBE_EXAMPLES = 3

probe_results, probe_summary = pilot.run_hf_grounding_probe(
    df,
    n_examples=N_PROBE_EXAMPLES,
    model_id=PROBE_MODEL_ID,
    max_new_tokens=160,
    max_evidence_chars=6000,
)

probe_summary_reset = probe_summary.reset_index()
probe_preview_cols = ['condition', 'target_answer', 'parsed_answer', 'probe_success', 'weak_match_answer', 'numeric_match_answer', 'refusal', 'prediction']
if 'compression_source' in probe_results.columns:
    probe_preview_cols.insert(1, 'compression_source')
probe_preview = probe_results[probe_preview_cols].head(20)

print('Probe conditions:', sorted(probe_results['condition'].unique().tolist()))
print('=== PROBE SUMMARY ===')
print(probe_summary_reset.to_string(index=False))
print('\n=== PROBE PREVIEW ===')
print(probe_preview.to_string(index=False))

display(probe_summary)
display(probe_preview)

probe_summary_reset.to_csv('probe_summary.csv', index=False)
probe_results.to_csv('probe_results.csv', index=False)
print('\nWrote probe_summary.csv and probe_results.csv')


## 7. Export Pilot Files


In [ ]:
csv_path, jsonl_path = pilot.export_pilot_files(df)
print(csv_path, jsonl_path)